# 22 — Edge Analysis

Where do edges appear? Analyze by time-in-game, score differential, game situation. Identify if there are systematic patterns the market gets wrong.

In [ ]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_trade_dates, load_trades, load_boxscores_for_game, discover_game_ids
from nba_edge.data.ticker_parser import parse_ticker, is_game_winner
from nba_edge.data.game_alignment import match_ticker_to_game
from nba_edge.backtest.engine import BacktestEngine, TradeWithEdge
from nba_edge.models.analytical import AnalyticalWinProb

In [ ]:
# Load and run backtest (reuse logic from notebook 21)
dates = discover_trade_dates(min_date='2026-04-21')
df = load_trades(dates)
engine = BacktestEngine(model=AnalyticalWinProb(variance_rate=0.44))

game_tickers = [t for t in df['market_ticker'].unique().to_list() if is_game_winner(t)]
results = []

for ticker in sorted(game_tickers):
    parsed = parse_ticker(ticker)
    if not parsed.game_date:
        continue
    try:
        available = discover_game_ids(parsed.game_date)
    except Exception:
        continue
    matched = match_ticker_to_game(ticker, available)
    if not matched:
        continue
    snapshots = load_boxscores_for_game(matched, parsed.game_date)
    if len(snapshots) < 50 or snapshots[-1].get('game_status', 0) != 3:
        continue
    ticker_trades = df.filter(pl.col('market_ticker') == ticker)
    result = engine.run_game(ticker_trades, snapshots, ticker)
    if result and len(result.trades) > 0:
        result.game_id = matched
        results.append(result)

# Flatten all trades
all_trades = [(t, r.outcome_yes) for r in results for t in r.trades]
print(f'Total trades across {len(results)} markets: {len(all_trades):,}')

In [ ]:
# Edge by time remaining (binned into 5-minute windows)
time_bins = list(range(0, 2881, 300))  # 0 to 2880 in 300s (5min) increments
bin_labels = [f'{b//60}-{(b+300)//60}min' for b in time_bins[:-1]]

bin_edges_mean = []
bin_edges_std = []
bin_counts = []

for i in range(len(time_bins) - 1):
    lo, hi = time_bins[i], time_bins[i+1]
    trades_in_bin = [t.edge * 100 for t, _ in all_trades if lo <= t.seconds_remaining < hi]
    if trades_in_bin:
        bin_edges_mean.append(np.mean(trades_in_bin))
        bin_edges_std.append(np.std(trades_in_bin))
        bin_counts.append(len(trades_in_bin))
    else:
        bin_edges_mean.append(0)
        bin_edges_std.append(0)
        bin_counts.append(0)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(bin_labels, bin_edges_mean, yerr=bin_edges_std, alpha=0.7, capsize=3)
axes[0].axhline(0, color='black', linestyle=':')
axes[0].set_ylabel('Mean Edge (cents)')
axes[0].set_title('Mean Edge by Time Remaining')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(bin_labels, bin_counts, color='steelblue', alpha=0.7)
axes[1].set_ylabel('Number of Trades')
axes[1].set_xlabel('Time Remaining')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Edge by absolute score differential
diff_bins = list(range(0, 31, 3))  # 0 to 30 in steps of 3

diff_edge_mean = []
diff_counts = []

for i in range(len(diff_bins) - 1):
    lo, hi = diff_bins[i], diff_bins[i+1]
    trades_in_bin = [t.edge * 100 for t, _ in all_trades if lo <= abs(t.score_diff) < hi]
    if trades_in_bin:
        diff_edge_mean.append(np.mean(np.abs(trades_in_bin)))
        diff_counts.append(len(trades_in_bin))
    else:
        diff_edge_mean.append(0)
        diff_counts.append(0)

labels = [f'{diff_bins[i]}-{diff_bins[i+1]}' for i in range(len(diff_bins)-1)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(labels, diff_edge_mean, color='darkorange', alpha=0.7)
ax.set_xlabel('Absolute Score Differential')
ax.set_ylabel('Mean |Edge| (cents)')
ax.set_title('Edge Magnitude by Score Differential')
plt.tight_layout()
plt.show()

In [ ]:
# Edge by period
period_edges = {}
for t, _ in all_trades:
    p = t.period
    if p not in period_edges:
        period_edges[p] = []
    period_edges[p].append(t.edge * 100)

print('Edge by period:')
for p in sorted(period_edges.keys()):
    edges = period_edges[p]
    print(f'  Q{p}: mean={np.mean(edges):+.2f}c, |mean|={np.mean(np.abs(edges)):.2f}c, n={len(edges)}')

In [ ]:
# When the model and market disagree most, who is right?
# Split into: model says higher prob vs market says higher prob
model_bullish = [(t, outcome) for t, outcome in all_trades if t.edge > 0.05]
model_bearish = [(t, outcome) for t, outcome in all_trades if t.edge < -0.05]

if model_bullish:
    bull_correct = sum(1 for t, o in model_bullish if o) / len(model_bullish)
    bull_avg_model = np.mean([t.model_prob_yes for t, _ in model_bullish])
    bull_avg_market = np.mean([t.market_implied for t, _ in model_bullish])
    print(f'Model bullish (edge > +5c): {len(model_bullish)} trades')
    print(f'  Actual YES rate: {bull_correct:.1%}')
    print(f'  Model predicted: {bull_avg_model:.1%}')
    print(f'  Market implied:  {bull_avg_market:.1%}')
    print(f'  → {"Model was right" if bull_correct > bull_avg_market else "Market was right"}')

print()
if model_bearish:
    bear_correct = sum(1 for t, o in model_bearish if not o) / len(model_bearish)
    bear_avg_model = np.mean([1 - t.model_prob_yes for t, _ in model_bearish])
    bear_avg_market = np.mean([1 - t.market_implied for t, _ in model_bearish])
    print(f'Model bearish (edge < -5c): {len(model_bearish)} trades')
    print(f'  Actual NO rate: {bear_correct:.1%}')
    print(f'  Model predicted NO: {bear_avg_model:.1%}')
    print(f'  Market implied NO:  {bear_avg_market:.1%}')
    print(f'  → {"Model was right" if bear_correct > bear_avg_market else "Market was right"}')

In [ ]:
# Hypothesis: edge is largest right after scoring runs (momentum)
# Check if edge correlates with recent score changes
# (proxy: large |score_diff| changes between adjacent trades)
print('\n--- Momentum analysis ---')
print('Do edges cluster after rapid score changes?')
print('(Requires further analysis with play-by-play data in future notebooks)')